# Локальный запуск DLightSB и DLightSB-MM

Ноутбук независимо запускает baseline и MM для размерностей 2 16 и 64

После каждого train и test выполните следующую ячейку чтобы сохранить отдельный ZIP-архив модели

Ничего не скачивается и не устанавливается


## Локальное окружение

Перед запуском выберите kernel окружения проекта

Для GPU-запуска CUDA должна быть доступна из этого kernel

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import csv
import hashlib
import json
import os
import subprocess
import sys
import zipfile


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "config.yaml").is_file() and (
            candidate / "src" / "run.py"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Не найден корень catsbench запустите ноутбук из папки проекта или notebooks"
    )


PROJECT_DIR = find_project_root(Path.cwd().resolve())
RUNS_DIR = PROJECT_DIR / "logs" / "runs"
EXPORT_DIR = PROJECT_DIR / "exports"
SEED = 42
RUNS = {}

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

print("Проект:", PROJECT_DIR)
print("Python:", sys.executable)
print("Логи:", RUNS_DIR)
print("Архивы:", EXPORT_DIR)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна в выбранном kernel выберите CUDA-окружение проекта"
    )

print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## TensorBoard

Эту ячейку можно выполнить до или во время обучения

In [ ]:
from IPython import get_ipython


TENSORBOARD_LOG_DIR = RUNS_DIR.resolve()
tensorboard_args = (
    f'--logdir "{TENSORBOARD_LOG_DIR.as_posix()}" '
    '--reload_interval 5'
)

ipython = get_ipython()
ipython.run_line_magic("load_ext", "tensorboard")
ipython.run_line_magic("tensorboard", tensorboard_args)

## Функции запуска

Каждый вызов выполняет обучение затем test последнего checkpoint

Baseline и MM явно запускаются на одной GPU

In [ ]:
def run_streaming(command: list[str], env: dict[str, str]) -> None:
    print("Команда:", subprocess.list2cmdline(command))

    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

In [ ]:
def experiment_name(kind: str, dim: int) -> str:
    if kind == "baseline":
        return f"dlight_sb/benchmark_hd/d{dim}_g002"

    if kind == "mm":
        return f"dlight_sb_mm/benchmark_hd/d{dim}_g002"

    raise ValueError(f"Неизвестный алгоритм {kind}")


def latest_completed_run(experiment: str) -> Path:
    seed_dir = RUNS_DIR / experiment / str(SEED)
    candidates = [
        path
        for path in seed_dir.glob("*")
        if path.is_dir() and (path / "checkpoints" / "last.ckpt").is_file()
    ]

    if not candidates:
        raise FileNotFoundError(
            f"Не найден завершённый запуск с last.ckpt в {seed_dir}"
        )

    return max(candidates, key=lambda path: path.name)


def run_experiment(kind: str, dim: int) -> Path:
    experiment = experiment_name(kind, dim)
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"

    if kind == "mm":
        thread_count = str(os.cpu_count() or 1)
        env.setdefault("OMP_NUM_THREADS", thread_count)
        env.setdefault("MKL_NUM_THREADS", thread_count)
        env.setdefault("OPENBLAS_NUM_THREADS", thread_count)

    common_overrides = [
        f"experiment={experiment}",
        "trainer.accelerator=gpu",
        "trainer.devices=1",
        "logger=csv_tensorboard",
        "++callbacks.plotter_callback=null",
    ]

    train_command = [
        sys.executable,
        "-m",
        "src.run",
        *common_overrides,
        "logger.csv.version=train",
    ]
    test_command = [
        sys.executable,
        "-m",
        "src.run",
        "task_name=test",
        "ckpt_path=auto",
        *common_overrides,
        "logger.csv.version=test",
    ]

    started = perf_counter()
    print(f"Запуск {kind} d={dim}")
    run_streaming(train_command, env)
    run_streaming(test_command, env)

    run_dir = latest_completed_run(experiment)
    RUNS.setdefault(dim, {})[kind] = run_dir
    elapsed = perf_counter() - started
    print(f"Готово за {elapsed / 60:.1f} минут")
    print("Результат:", run_dir)
    return run_dir


## Функции экспорта

Архив создаётся отдельно для каждой модели в каталоге `exports`

Его можно загрузить в Colab-анализ или выбрать в локальном ноутбуке анализа


In [ ]:
def sha256sum(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def find_metrics_csv(run_dir: Path, stage: str) -> Path:
    preferred = run_dir / "csv" / stage / "metrics.csv"

    if preferred.is_file():
        return preferred

    matches = sorted(run_dir.glob(f"*/{stage}/metrics.csv"))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Ожидался один {stage}/metrics.csv в {run_dir} найдено {len(matches)}"
        )

    return matches[0]


def validate_run(run_dir: Path) -> dict:
    checkpoint = run_dir / "checkpoints" / "last.ckpt"
    train_metrics = find_metrics_csv(run_dir, "train")
    test_metrics = find_metrics_csv(run_dir, "test")

    required = [
        checkpoint,
        train_metrics,
        test_metrics,
        run_dir / ".hydra" / "config.yaml",
        run_dir / ".hydra" / "overrides.yaml",
        run_dir / "train.log",
        run_dir / "test.log",
    ]
    missing = [str(path) for path in required if not path.is_file()]

    if missing:
        raise FileNotFoundError(
            "В запуске отсутствуют обязательные файлы:\n" + "\n".join(missing)
        )

    with train_metrics.open(encoding="utf-8", newline="") as file:
        columns = next(csv.reader(file))

    validation_columns = [
        column
        for column in columns
        if column.startswith("val/")
    ]

    if not validation_columns:
        raise RuntimeError(f"В {train_metrics} нет validation-метрик")

    return {
        "run_dir": str(run_dir),
        "last_checkpoint": str(checkpoint.relative_to(run_dir)),
        "last_checkpoint_bytes": checkpoint.stat().st_size,
        "last_checkpoint_sha256": sha256sum(checkpoint),
        "train_metrics": str(train_metrics.relative_to(run_dir)),
        "test_metrics": str(test_metrics.relative_to(run_dir)),
        "validation_columns": validation_columns,
    }


def add_directory_to_zip(
    archive: zipfile.ZipFile,
    source_dir: Path,
    prefix: str,
) -> None:
    for path in sorted(source_dir.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(Path(prefix) / path.relative_to(source_dir)),
            )


def git_value(*arguments: str) -> str:
    return subprocess.run(
        ["git", *arguments],
        cwd=PROJECT_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


def export_run(kind: str, dim: int) -> Path:
    if kind not in {"baseline", "mm"}:
        raise ValueError(f"Неизвестный тип эксперимента: {kind}")

    run_dir = RUNS.get(dim, {}).get(kind)

    if run_dir is None or not Path(run_dir).is_dir():
        run_dir = latest_completed_run(experiment_name(kind, dim))

    run_dir = Path(run_dir)
    run_info = validate_run(run_dir)
    created_at = datetime.now(timezone.utc)
    timestamp = created_at.strftime("%Y-%m-%d_%H-%M-%S_UTC")
    archive_path = EXPORT_DIR / f"catsbench_d{dim}_g002_{kind}_{timestamp}.zip"

    manifest = {
        "archive_format": "single_run_v1",
        "created_at_utc": created_at.isoformat(),
        "repository": git_value("config", "--get", "remote.origin.url"),
        "branch": git_value("branch", "--show-current"),
        "commit": git_value("rev-parse", "HEAD"),
        "seed": SEED,
        "dimension": dim,
        "kind": kind,
        "hydra_experiment": experiment_name(kind, dim),
        "run": run_info,
    }

    with zipfile.ZipFile(
        archive_path,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:
        archive.writestr(
            "manifest.json",
            json.dumps(manifest, ensure_ascii=False, indent=2),
        )
        add_directory_to_zip(archive, run_dir, kind)

    print("Архив готов:", archive_path)
    print(f"Размер: {archive_path.stat().st_size / 1024**2:.1f} MiB")
    print(f"Финальный checkpoint: {kind}/checkpoints/last.ckpt")
    return archive_path


# Размерность 2

## Baseline DLightSB d2

In [ ]:
baseline_d2_run = run_experiment("baseline", 2)


### Сохранить baseline d2


In [ ]:
baseline_d2_archive = export_run("baseline", 2)


## DLightSB-MM d2

In [ ]:
mm_d2_run = run_experiment("mm", 2)


### Сохранить DLightSB-MM d2


In [ ]:
mm_d2_archive = export_run("mm", 2)


# Размерность 16

## Baseline DLightSB d16

In [ ]:
baseline_d16_run = run_experiment("baseline", 16)


### Сохранить baseline d16


In [ ]:
baseline_d16_archive = export_run("baseline", 16)


## DLightSB-MM d16

In [ ]:
mm_d16_run = run_experiment("mm", 16)


### Сохранить DLightSB-MM d16


In [ ]:
mm_d16_archive = export_run("mm", 16)


# Размерность 64

## Baseline DLightSB d64

In [ ]:
baseline_d64_run = run_experiment("baseline", 64)


### Сохранить baseline d64


In [ ]:
baseline_d64_archive = export_run("baseline", 64)


## DLightSB-MM d64

In [ ]:
mm_d64_run = run_experiment("mm", 64)


### Сохранить DLightSB-MM d64


In [ ]:
mm_d64_archive = export_run("mm", 64)


## Локальные результаты

Полные каталоги запусков остаются в `logs/runs`

Отдельные переносимые архивы моделей сохраняются в `exports`

Каждый ZIP содержит checkpoint CSV TensorBoard Hydra-конфиги и train/test логи одной модели
